# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze a dataset defined by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The FAIR^2 dataset is described by a Croissant schema at:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
We load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset and inspect its metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}\n")
print("Version:", metadata.version)
print("Published on:", metadata.datePublished)
print("Authors:", metadata.author)
print("License:", metadata.license)

## 2. Data Overview
Let's review the available record sets in the dataset using their `@id` fields, along with the fields (columns) each contains, again referencing their `@id`s.

The metadata may contain multiple record sets describing survey data, coefficients, or log likelihood traces.

In [ ]:
# List record sets by their @id and show their fields
record_sets = list(dataset.record_sets)
print("Available record sets:")
for rs in record_sets:
    print(f"  @id: {rs['@id']}")
    print(f"   label: {rs.get('label', rs.get('name', ''))}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("   Fields:")
    for fld in fields:
        if isinstance(fld, dict):
            print(f"    - @id: {fld['@id']}, name: {fld.get('name', fld.get('label', ''))}, dataType: {fld.get('dataType', '')}")
        else:
            print(f"    - @id: {fld}")
    print()

## 3. Data Extraction
We'll load data from each record set into a pandas DataFrame for further analysis. We'll use ONLY the `@id` of each record set to reference it.

Replace or extend the `record_set_ids` below as appropriate for your dataset.

In [ ]:
# Gather record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set @id: {rs_id} with shape {dataframes[rs_id].shape}")
    except Exception as e:
        print(f"Could not load records for @id: {rs_id}: {e}")
        continue

# List DataFrame columns for the first available record set with data
first_available_rs = next((rs_id for rs_id, df in dataframes.items() if not df.empty), None)

if first_available_rs:
    print(f"\nColumns for record set @id {first_available_rs}:")
    print(dataframes[first_available_rs].columns.tolist())
    display(dataframes[first_available_rs].head())
else:
    print("No non-empty DataFrames loaded.")

## 4. Exploratory Data Analysis (EDA)
We'll apply basic EDA: filter by a numeric field, normalize it, and group by another field. All field references use `@id`s.

*Update the numeric field and group field `@id` below depending on your data!*

In [ ]:
# Select a record set and numeric field for analysis
record_set_id = first_available_rs  # Use the first available record set with data
df = dataframes[record_set_id]

# Identify a numeric field by checking dtypes
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field: {numeric_field_id}")
else:
    print("No numeric field found for EDA.")
    numeric_field_id = None

if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df = filtered_df.copy()
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Try to group by a categorical field (choose first object column different from numeric_field_id)
    group_fields = [col for col in df.columns if col != numeric_field_id and df[col].dtype == object]
    if group_fields:
        group_field_id = group_fields[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No categorical/group field found for grouping.")

## 5. Visualization
Visualize the distribution and relationships in the data. Only run if numeric field is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    # If grouping field is available, plot group comparison
    if group_fields:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR² dataset "Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya" using `mlcroissant`.

- We listed all available record sets and fields (referencing their `@id`s);<br>
- We loaded the data into DataFrames, performed basic filtering and normalization on numeric fields, and visualized their distributions.

This structure can be adapted and extended for deeper domain-specific analysis as needed, using the dataset's Croissant schema as a reliable, machine-actionable map.